# NASA C-MAPSS FD001 — RUL Prediction (1D-CNN + BiLSTM + Attention, 5-Fold Ensemble)
**Target:** RMSE < 11.0 on FD001  
**Architecture:** Global StandardScaler → 1D-CNN + BiLSTM + Multi-head Attention → 5-fold ensemble  
**Key improvements:** global normalization (train-fitted), sensor selection, piecewise-linear RUL, cosine annealing w/ warm restarts, Huber loss, deep ensemble

In [ ]:
# ============================================================
# Runtime compatibility shim for Kaggle GPU accelerators
# ============================================================
import subprocess, sys, os

def _gpu_name():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']
        ).decode()
        return out.strip().splitlines()[0] if out.strip() else ''
    except Exception:
        return ''

_GPU = _gpu_name()
print(f'Detected GPU: {_GPU!r}')

if 'P100' in _GPU:
    print('Tesla P100 detected — installing torch 2.3.1+cu118 for sm_60 support...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.3.1', 'torchvision==0.18.1',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
    ])
    for _m in list(sys.modules.keys()):
        if _m == 'torch' or _m.startswith('torch.'):
            del sys.modules[_m]

import torch as _t
print(f'torch: {_t.__version__}  |  cuda ok: {_t.cuda.is_available()}')

In [ ]:
"""
NASA C-MAPSS Turbofan Engine — Remaining Useful Life (RUL) Prediction
======================================================================
Architecture : 1D-CNN + BiLSTM + Multi-head Self-Attention (5-fold ensemble)
Normalization: Global StandardScaler (fit on train, apply to both)
RUL Target   : Piecewise-linear, capped at 125 cycles
Loss         : Huber (robust to outliers)
Scheduler    : CosineAnnealingWarmRestarts
Dataset      : NASA C-MAPSS FD001
"""

import os, json, math, time, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================
SEED          = 42
SUBSET        = 'FD001'
SEQ_LEN       = 30        # sliding window length
RUL_CAP       = 125       # piecewise-linear RUL cap

# Model
N_CNN_FILTERS = 64
CNN_KERNEL    = 3
LSTM_HIDDEN   = 128
N_LSTM_LAYERS = 2
N_HEADS       = 4
D_FF          = 256
DROPOUT       = 0.3        # increased from 0.2

# Training
BATCH_SIZE    = 256
N_EPOCHS      = 200
LR            = 1e-3
WEIGHT_DECAY  = 5e-4       # increased from 1e-4
PATIENCE      = 40         # increased from 30
T0            = 30         # cosine annealing period
T_MULT        = 2          # cosine annealing multiplier
N_FOLDS       = 5
HUBER_DELTA   = 10.0

# Noise augmentation
NOISE_STD     = 0.01       # increased from 0.005

# Sensors to DROP (constant / near-constant in FD001)
DROP_SENSORS  = ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
# Also drop operational settings (not informative for FD001 single regime)
DROP_OPS      = ['op_1', 'op_2', 'op_3']

def _find_cmapss_dir():
    for root in [Path('/kaggle/input'), Path('./data'), Path('../../data')]:
        if not root.exists():
            continue
        for p in root.rglob('train_FD001.txt'):
            return p.parent
    return Path('./data')

DATA_DIR = _find_cmapss_dir()
OUT_DIR  = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATA_DIR : {DATA_DIR}')
print(f'OUT_DIR  : {OUT_DIR}')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================
COL_NAMES = (
    ['unit', 'cycle'] +
    [f'op_{i}' for i in range(1, 4)] +
    [f's_{i}'  for i in range(1, 22)]
)

# Useful sensor columns after dropping constant sensors and ops
SENSOR_COLS = [
    c for c in COL_NAMES
    if c not in ['unit', 'cycle'] + DROP_SENSORS + DROP_OPS
]
N_FEATURES = len(SENSOR_COLS)
print(f'Useful features: {N_FEATURES} => {SENSOR_COLS}')

def load_raw(subset=SUBSET):
    train_df = pd.read_csv(
        DATA_DIR / f'train_{subset}.txt',
        sep=r'\s+', header=None, names=COL_NAMES
    )
    test_df = pd.read_csv(
        DATA_DIR / f'test_{subset}.txt',
        sep=r'\s+', header=None, names=COL_NAMES
    )
    rul_df = pd.read_csv(
        DATA_DIR / f'RUL_{subset}.txt',
        sep=r'\s+', header=None, names=['RUL']
    )
    return train_df, test_df, rul_df

train_raw, test_raw, rul_df = load_raw(SUBSET)
print(f'Train shape: {train_raw.shape}')
print(f'Test  shape: {test_raw.shape}')
print(f'Engines train: {train_raw["unit"].nunique()}, test: {test_raw["unit"].nunique()}')

In [ ]:
# ============================================================
# GLOBAL NORMALIZATION (fit on train, apply to both)
# ============================================================
# IMPORTANT: Per-engine normalization causes distribution mismatch
# between train (full trajectories) and test (partial trajectories).
# Global normalization fit on training data is the correct approach
# for generalizable test predictions.

from sklearn.preprocessing import StandardScaler

global_scaler = StandardScaler()
global_scaler.fit(train_raw[SENSOR_COLS].values)

train_norm = train_raw.copy()
test_norm  = test_raw.copy()

train_norm[SENSOR_COLS] = global_scaler.transform(train_raw[SENSOR_COLS].values)
test_norm[SENSOR_COLS]  = global_scaler.transform(test_raw[SENSOR_COLS].values)

print('Global StandardScaler normalization done (fit on train, applied to both).')
print(f'Train sensor range: [{train_norm[SENSOR_COLS].min().min():.3f}, {train_norm[SENSOR_COLS].max().max():.3f}]')
print(f'Test  sensor range: [{test_norm[SENSOR_COLS].min().min():.3f}, {test_norm[SENSOR_COLS].max().max():.3f}]')

In [ ]:
# ============================================================
# PIECEWISE-LINEAR RUL TARGET
# ============================================================
def add_rul_piecewise(df, cap=RUL_CAP):
    """
    Standard piecewise-linear RUL:
      - constant at `cap` when far from failure
      - linearly decreasing toward 0 at end of life
    This matches the assumption that early operation is healthy
    and degradation only becomes informative in the last `cap` cycles.
    """
    df = df.copy()
    max_cycle = df.groupby('unit')['cycle'].transform('max')
    raw_rul   = max_cycle - df['cycle']
    df['RUL'] = raw_rul.clip(upper=cap).astype(np.float32)
    return df


train_labeled = add_rul_piecewise(train_norm, cap=RUL_CAP)

# Test RUL ground truth (capped)
y_test_true = np.minimum(
    rul_df['RUL'].values.astype(np.float32), RUL_CAP
)

print(f'RUL distribution (train):')
print(f'  min={train_labeled["RUL"].min():.1f}, '
      f'max={train_labeled["RUL"].max():.1f}, '
      f'mean={train_labeled["RUL"].mean():.1f}')
print(f'RUL distribution (test, capped):')
print(f'  min={y_test_true.min():.1f}, '
      f'max={y_test_true.max():.1f}, '
      f'mean={y_test_true.mean():.1f}')

In [ ]:
# ============================================================
# SLIDING WINDOW DATASET
# ============================================================
def make_windows_train(df, feature_cols, seq_len=SEQ_LEN):
    """
    Extract all possible sliding windows from training data.
    Returns X: (N, seq_len, n_features), y: (N,)
    """
    X_list, y_list = [], []
    for _, grp in df.groupby('unit'):
        feats = grp[feature_cols].values.astype(np.float32)
        ruls  = grp['RUL'].values.astype(np.float32)
        n     = len(feats)
        for start in range(n - seq_len + 1):
            X_list.append(feats[start:start + seq_len])
            y_list.append(ruls[start + seq_len - 1])
    return np.stack(X_list, axis=0), np.array(y_list, dtype=np.float32)


def make_windows_test(test_df, feature_cols, seq_len=SEQ_LEN):
    """
    One window per test engine: the LAST seq_len timesteps.
    If engine has fewer timesteps, zero-pad at the front.
    """
    X_list = []
    for _, grp in test_df.groupby('unit'):
        feats = grp[feature_cols].values.astype(np.float32)
        n     = len(feats)
        if n >= seq_len:
            X_list.append(feats[-seq_len:])
        else:
            pad   = np.zeros((seq_len - n, len(feature_cols)), dtype=np.float32)
            X_list.append(np.vstack([pad, feats]))
    return np.stack(X_list, axis=0)


X_all, y_all = make_windows_train(train_labeled, SENSOR_COLS)
X_test_np    = make_windows_test(test_norm, SENSOR_COLS)

print(f'Train windows : {X_all.shape}  |  labels: {y_all.shape}')
print(f'Test  windows : {X_test_np.shape}')
print(f'Test  true RUL: {y_test_true.shape}')

In [ ]:
# ============================================================
# DATASET & DATALOADER
# ============================================================
class CMAPSSDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, augment: bool = False):
        self.X       = torch.from_numpy(X).float()
        self.y       = torch.from_numpy(y).float()
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.augment:
            # Add slight Gaussian noise for data augmentation
            x = x + torch.randn_like(x) * NOISE_STD
        return x, self.y[idx]


class TestDataset(Dataset):
    def __init__(self, X: np.ndarray):
        self.X = torch.from_numpy(X).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]


test_dataset = TestDataset(X_test_np)
test_loader  = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
print('DataLoader utilities ready.')

In [ ]:
# ============================================================
# MODEL: 1D-CNN + BiLSTM + Multi-head Attention
# ============================================================
class CNNBiLSTMAttention(nn.Module):
    """
    Architecture proven in multiple C-MAPSS papers to achieve RMSE < 11:
      1. 1D-CNN: extract local temporal patterns from sensor data
      2. BiLSTM: capture long-range dependencies in both directions
      3. Multi-head Self-Attention: focus on degradation-relevant timesteps
      4. Regression head: MLP with dropout
    """

    def __init__(
        self,
        n_features   : int,
        cnn_filters  : int  = N_CNN_FILTERS,
        cnn_kernel   : int  = CNN_KERNEL,
        lstm_hidden  : int  = LSTM_HIDDEN,
        n_lstm_layers: int  = N_LSTM_LAYERS,
        n_heads      : int  = N_HEADS,
        d_ff         : int  = D_FF,
        dropout      : float = DROPOUT,
    ):
        super().__init__()

        # ── 1D-CNN block ──────────────────────────────────────────
        # Input: (B, T, F)  → transpose → (B, F, T) for Conv1d
        self.cnn = nn.Sequential(
            nn.Conv1d(n_features, cnn_filters, kernel_size=cnn_kernel, padding=cnn_kernel // 2),
            nn.BatchNorm1d(cnn_filters),
            nn.GELU(),
            nn.Conv1d(cnn_filters, cnn_filters, kernel_size=cnn_kernel, padding=cnn_kernel // 2),
            nn.BatchNorm1d(cnn_filters),
            nn.GELU(),
            nn.Conv1d(cnn_filters, cnn_filters * 2, kernel_size=cnn_kernel, padding=cnn_kernel // 2),
            nn.BatchNorm1d(cnn_filters * 2),
            nn.GELU(),
        )
        cnn_out_ch = cnn_filters * 2  # output channels after CNN

        # ── BiLSTM ────────────────────────────────────────────────
        self.bilstm = nn.LSTM(
            input_size   = cnn_out_ch,
            hidden_size  = lstm_hidden,
            num_layers   = n_lstm_layers,
            batch_first  = True,
            bidirectional= True,
            dropout      = dropout if n_lstm_layers > 1 else 0.0,
        )
        bilstm_out = lstm_hidden * 2

        # ── Multi-head Self-Attention ──────────────────────────────
        self.attn_norm = nn.LayerNorm(bilstm_out)
        self.mha       = nn.MultiheadAttention(
            embed_dim  = bilstm_out,
            num_heads  = n_heads,
            dropout    = dropout,
            batch_first= True,
        )
        # Position-wise feed-forward (Pre-LN Transformer style)
        self.ff_norm = nn.LayerNorm(bilstm_out)
        self.ff      = nn.Sequential(
            nn.Linear(bilstm_out, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, bilstm_out),
        )

        # ── Regression head ───────────────────────────────────────
        # Combine: last BiLSTM hidden + attention-weighted mean pool
        head_in = bilstm_out * 2
        self.regressor = nn.Sequential(
            nn.Linear(head_in, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'weight_hh' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)

    def forward(self, x):          # x: (B, T, F)
        # 1D-CNN
        h = self.cnn(x.transpose(1, 2))   # (B, C, T)
        h = h.transpose(1, 2)             # (B, T, C)

        # BiLSTM
        lstm_out, (hn, _) = self.bilstm(h)  # (B, T, 2H), hn: (2*L, B, H)
        # Concatenate final forward + backward hidden states
        last_h = torch.cat([hn[-2], hn[-1]], dim=-1)  # (B, 2H)

        # Multi-head self-attention (Pre-LN)
        residual  = lstm_out
        normed    = self.attn_norm(lstm_out)
        attn_out, _ = self.mha(normed, normed, normed)
        lstm_out  = residual + attn_out

        # Position-wise feed-forward
        residual  = lstm_out
        normed    = self.ff_norm(lstm_out)
        lstm_out  = residual + self.ff(normed)

        # Global average pool over time (attention-refined)
        mean_pool = lstm_out.mean(dim=1)   # (B, 2H)

        # Concatenate and regress
        out = torch.cat([last_h, mean_pool], dim=-1)  # (B, 4H)
        return self.regressor(out).squeeze(-1)         # (B,)


_test_model = CNNBiLSTMAttention(N_FEATURES).to(DEVICE)
_n_params   = sum(p.numel() for p in _test_model.parameters())
print(f'Model parameters: {_n_params:,}')

# Sanity forward pass
_dummy = torch.zeros(4, SEQ_LEN, N_FEATURES).to(DEVICE)
_out   = _test_model(_dummy)
print(f'Output shape: {_out.shape}  (expected [4])')
del _test_model, _dummy, _out

In [ ]:
# ============================================================
# TRAINING UTILITIES
# ============================================================
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def nasa_score(y_true, y_pred):
    d = np.asarray(y_pred) - np.asarray(y_true)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return float(s.sum())


def train_epoch(model, loader, optimiser, criterion, grad_scaler):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)
        optimiser.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        grad_scaler.step(optimiser)
        grad_scaler.update()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    preds_list, labels_list = [], []
    total_loss = 0.0
    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            X_batch, y_batch = batch
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            pred    = model(X_batch)
            if criterion is not None:
                total_loss += criterion(pred, y_batch).item() * X_batch.size(0)
            labels_list.append(y_batch.cpu().numpy())
        else:
            X_batch = batch.to(DEVICE)
            pred    = model(X_batch)
        preds_list.append(pred.cpu().numpy())
    preds = np.concatenate(preds_list)
    if labels_list:
        labels = np.concatenate(labels_list)
        avg_loss = total_loss / sum(len(b) for b in labels_list)
        return avg_loss, preds, labels
    return preds


print('Training utilities defined.')

In [ ]:
# ============================================================
# 5-FOLD CROSS-VALIDATION TRAINING
# ============================================================
# We split on ENGINE ids (not on windows) to avoid data leakage
train_units = train_labeled['unit'].unique()
kf          = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_test_preds   = []   # each fold's test predictions
fold_val_rmses    = []
fold_histories    = []

best_model_states = []   # best state_dict per fold

t_start = time.time()

for fold_idx, (tr_units_idx, va_units_idx) in enumerate(
        kf.split(train_units), start=1
):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold_idx}/{N_FOLDS}')
    print(f'{"="*60}')

    tr_units_fold = train_units[tr_units_idx]
    va_units_fold = train_units[va_units_idx]

    # Rebuild windows for this fold's train / val engines
    train_fold_df = train_labeled[train_labeled['unit'].isin(tr_units_fold)]
    val_fold_df   = train_labeled[train_labeled['unit'].isin(va_units_fold)]

    X_tr_f, y_tr_f = make_windows_train(train_fold_df, SENSOR_COLS)
    X_va_f, y_va_f = make_windows_train(val_fold_df,   SENSOR_COLS)

    train_ds = CMAPSSDataset(X_tr_f, y_tr_f, augment=True)
    val_ds   = CMAPSSDataset(X_va_f, y_va_f, augment=False)

    train_ld = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True
    )
    val_ld = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    )

    # Fresh model for each fold
    torch.manual_seed(SEED + fold_idx)
    model     = CNNBiLSTMAttention(N_FEATURES).to(DEVICE)
    optimiser = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    # Cosine Annealing with Warm Restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimiser, T_0=T0, T_mult=T_MULT, eta_min=LR * 0.01
    )
    criterion  = nn.HuberLoss(delta=HUBER_DELTA)
    grad_scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    best_val_rmse = float('inf')
    best_state    = None
    patience_cnt  = 0
    hist          = {'train_loss': [], 'val_loss': [], 'val_rmse': [], 'lr': []}

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss = train_epoch(model, train_ld, optimiser, criterion, grad_scaler)
        val_loss, val_preds, val_gt = evaluate(model, val_ld, criterion)
        scheduler.step(epoch)
        curr_lr  = optimiser.param_groups[0]['lr']
        val_r    = rmse(val_gt, val_preds)

        hist['train_loss'].append(tr_loss)
        hist['val_loss'].append(val_loss)
        hist['val_rmse'].append(val_r)
        hist['lr'].append(curr_lr)

        if val_r < best_val_rmse:
            best_val_rmse = val_r
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
            star          = '  ***'
        else:
            patience_cnt += 1
            star          = ''

        if epoch % 10 == 0 or star:
            print(f'  Ep {epoch:3d}/{N_EPOCHS} | '
                  f'tr_loss={tr_loss:.4f} | '
                  f'val_loss={val_loss:.4f} | '
                  f'val_rmse={val_r:.4f} | '
                  f'lr={curr_lr:.6f}'
                  f'{star}')

        if patience_cnt >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}')
            break

    # Collect results for this fold
    best_model_states.append(best_state)
    fold_val_rmses.append(best_val_rmse)
    fold_histories.append(hist)

    # Predict test set with best model
    model.load_state_dict(best_state)
    fold_test_pred = evaluate(model, test_loader)
    fold_test_preds.append(fold_test_pred)

    fold_test_r = rmse(y_test_true, fold_test_pred)
    print(f'  Fold {fold_idx} | best_val_rmse={best_val_rmse:.4f} | test_rmse={fold_test_r:.4f}')

elapsed = time.time() - t_start
print(f'\nTotal training time: {elapsed:.1f}s')
print(f'Val RMSE per fold: {[f"{v:.4f}" for v in fold_val_rmses]}')
print(f'Mean val RMSE: {np.mean(fold_val_rmses):.4f} ± {np.std(fold_val_rmses):.4f}')

In [ ]:
# ============================================================
# ENSEMBLE: AVERAGE FOLD PREDICTIONS
# ============================================================

# Simple mean ensemble of all folds
ensemble_preds = np.stack(fold_test_preds, axis=0).mean(axis=0)
ensemble_preds = np.clip(ensemble_preds, 0, RUL_CAP)

ensemble_rmse  = rmse(y_test_true, ensemble_preds)
ensemble_score = nasa_score(y_test_true, ensemble_preds)
ensemble_mae   = float(np.mean(np.abs(y_test_true - ensemble_preds)))

# Also check individual fold test RMSE
fold_test_rmses = [rmse(y_test_true, p) for p in fold_test_preds]
best_single_rmse = min(fold_test_rmses)

print('=' * 60)
print('FINAL RESULTS — C-MAPSS FD001')
print('=' * 60)
for i, (vr, tr) in enumerate(zip(fold_val_rmses, fold_test_rmses), 1):
    print(f'  Fold {i}: val_rmse={vr:.4f}, test_rmse={tr:.4f}')
print('-' * 60)
print(f'  Ensemble Test RMSE : {ensemble_rmse:.4f}')
print(f'  Ensemble Test MAE  : {ensemble_mae:.4f}')
print(f'  NASA Score         : {ensemble_score:.2f}')
print(f'  Best Single-Fold   : {best_single_rmse:.4f}')
print('=' * 60)

In [ ]:
# ============================================================
# SAVE METRICS
# ============================================================
metrics = {
    'subset'              : SUBSET,
    'test_rmse'           : round(ensemble_rmse, 4),
    'test_mae'            : round(ensemble_mae, 4),
    'nasa_score'          : round(ensemble_score, 2),
    'best_single_fold_rmse': round(best_single_rmse, 4),
    'fold_val_rmses'      : [round(v, 4) for v in fold_val_rmses],
    'fold_test_rmses'     : [round(v, 4) for v in fold_test_rmses],
    'mean_val_rmse'       : round(float(np.mean(fold_val_rmses)), 4),
    'training_time_s'     : round(elapsed, 1),
    'config': {
        'subset'         : SUBSET,
        'seq_len'        : SEQ_LEN,
        'rul_cap'        : RUL_CAP,
        'n_features'     : N_FEATURES,
        'n_folds'        : N_FOLDS,
        'epochs'         : N_EPOCHS,
        'batch_size'     : BATCH_SIZE,
        'lr'             : LR,
        'cnn_filters'    : N_CNN_FILTERS,
        'lstm_hidden'    : LSTM_HIDDEN,
        'n_lstm_layers'  : N_LSTM_LAYERS,
        'n_heads'        : N_HEADS,
        'dropout'        : DROPOUT,
        'huber_delta'    : HUBER_DELTA,
        'dropped_sensors': DROP_SENSORS,
        'scheduler'      : f'CosineAnnealingWarmRestarts(T0={T0}, T_mult={T_MULT})',
        'noise_std'      : NOISE_STD,
    },
}

metrics_path = OUT_DIR / 'metrics.json'
with open(metrics_path, 'w') as fh:
    json.dump(metrics, fh, indent=2)
print(f'Saved: {metrics_path}')
print(json.dumps(metrics, indent=2))

In [ ]:
# ============================================================
# VISUALISATION
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f'C-MAPSS FD001 — 1D-CNN + BiLSTM + Attention (5-Fold Ensemble)\n'
    f'Test RMSE: {ensemble_rmse:.4f} | NASA Score: {ensemble_score:.0f}',
    fontsize=13, fontweight='bold'
)

colors = plt.cm.tab10.colors

# Panel 1: Training loss curves (all folds)
ax = axes[0, 0]
for fi, hist in enumerate(fold_histories):
    ax.plot(hist['train_loss'], color=colors[fi], alpha=0.7, linewidth=1.2,
            label=f'Fold {fi+1}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Huber Loss')
ax.set_title('Train Loss (All Folds)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 2: Validation RMSE curves
ax = axes[0, 1]
for fi, hist in enumerate(fold_histories):
    ax.plot(hist['val_rmse'], color=colors[fi], alpha=0.7, linewidth=1.2,
            label=f'Fold {fi+1}')
ax.axhline(y=ensemble_rmse, color='red', linestyle='--', linewidth=1.5,
           label=f'Ensemble={ensemble_rmse:.2f}')
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
ax.set_title('Validation RMSE (All Folds)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 3: LR schedule (fold 1)
ax = axes[0, 2]
ax.plot(fold_histories[0]['lr'], color='purple', linewidth=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
ax.set_title('LR Schedule (CosineAnnealingWarmRestarts, Fold 1)')
ax.grid(alpha=0.3)

# Panel 4: Actual vs Predicted scatter
ax = axes[1, 0]
ax.scatter(y_test_true, ensemble_preds, alpha=0.7, s=50,
           c='steelblue', edgecolors='none')
lim = max(y_test_true.max(), ensemble_preds.max()) + 5
ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect')
ax.set_xlabel('Actual RUL'); ax.set_ylabel('Predicted RUL')
ax.set_title(f'Actual vs Predicted RUL\nRMSE={ensemble_rmse:.2f}')
ax.legend(); ax.grid(alpha=0.3)

# Panel 5: Sorted predictions comparison
ax = axes[1, 1]
order = np.argsort(y_test_true)
ax.plot(y_test_true[order], label='True RUL', linewidth=2, color='black')
ax.plot(ensemble_preds[order], label='Ensemble', linewidth=1.5,
        linestyle='--', color='red', alpha=0.8)
for fi, fp in enumerate(fold_test_preds):
    ax.plot(fp[order], alpha=0.3, linewidth=0.8, color=colors[fi])
ax.set_xlabel('Engine (sorted by true RUL)')
ax.set_ylabel('RUL (cycles)')
ax.set_title('True vs Predicted RUL (Sorted)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 6: Error histogram
ax = axes[1, 2]
errors = ensemble_preds - y_test_true
ax.hist(errors, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
ax.axvline(x=errors.mean(), color='orange', linestyle='-', linewidth=1.5,
           label=f'Mean={errors.mean():.2f}')
ax.set_xlabel('Prediction Error (pred - actual)')
ax.set_ylabel('Count')
ax.set_title('Error Distribution')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plot_path = OUT_DIR / 'training_results.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
print(f'Saved: {plot_path}')
plt.show()

In [ ]:
# ============================================================
# SAVE BEST FOLD MODEL
# ============================================================
best_fold_idx = int(np.argmin(fold_val_rmses))
best_state    = best_model_states[best_fold_idx]
model_path    = OUT_DIR / 'best_model.pt'
torch.save({
    'state_dict'   : best_state,
    'fold'         : best_fold_idx + 1,
    'val_rmse'     : fold_val_rmses[best_fold_idx],
    'n_features'   : N_FEATURES,
    'seq_len'      : SEQ_LEN,
    'sensor_cols'  : SENSOR_COLS,
    'config': {
        'cnn_filters'  : N_CNN_FILTERS,
        'cnn_kernel'   : CNN_KERNEL,
        'lstm_hidden'  : LSTM_HIDDEN,
        'n_lstm_layers': N_LSTM_LAYERS,
        'n_heads'      : N_HEADS,
        'd_ff'         : D_FF,
        'dropout'      : DROPOUT,
    },
}, model_path)
print(f'Best model (fold {best_fold_idx+1}) saved: {model_path}')

print()
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'  Test RMSE  : {ensemble_rmse:.4f}  (target < 11.0)')
print(f'  Test MAE   : {ensemble_mae:.4f}')
print(f'  NASA Score : {ensemble_score:.2f}')
print(f'  Target met : {"YES" if ensemble_rmse < 11.0 else "NO — needs further tuning"}')
print('=' * 60)